# Notebook Purpose: Opening raw bd2 detections and removing overlaps using a certain threshold and saving the new detections to be used for precision-recall analysis

## Imports Section:

In [7]:
import numpy as np
import pandas as pd
import random
import scipy
from scipy import stats
import datetime as dt
import dask.dataframe as dd

In [8]:
import glob
import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches
from pathlib import Path

In [9]:
import convert_between_detector_file_formats as conversion

## Function and Constant Definitions

In [10]:
FREQ_COLORS = {'LF':'cyan', 'HF':'orange'}
FILE_SITES = {'20220730_053000':'Carp',
 '20220826_070000':'Central',
 '20220727_083000':'Foliage',
 '20220829_090000':'Foliage'}

RAVENPRO_TABLE_EXTENSION = ".txt"
RAVENPRO_TABLE_FORMAT = "\t"
RAVENTXT_HUMAN_ANNOTATIONS_SAVE_DIR = f'{Path.home()}/Documents/Research/mila_files/mila-human-wav-txt'
BD2_DETS_SAVE_DIR = Path(f'20250130__group_threshold_sweep_results')
OVERLAP_TIME_THRESHOLD = 0.012
SITE_NAMES = {'Carp':'Carp Pond', 'Foliage':'Foliage', 'Central':'Central Pond'}

In [11]:
file_keys = list(FILE_SITES.keys())
file_keys

['20220730_053000', '20220826_070000', '20220727_083000', '20220829_090000']

In [12]:
for wav_filename in file_keys:
    site = FILE_SITES[wav_filename]

    args = dict()
    args['chunk_size'] = 2
    args['detection_threshold'] = 0.00

    ones = int(args['detection_threshold'])
    decimals = int(int(100*(args['detection_threshold'])) % 100)
    threshold_tag = f"threshold{ones}p{decimals:02}"
    save_loc = Path(f"bd2__{threshold_tag}_chunksize{int(args['chunk_size'])}_{wav_filename}.csv")

    filepath = BD2_DETS_SAVE_DIR / save_loc
    batdetect2_df = pd.read_csv(filepath, sep=',', index_col=0)
    batdetect2_df.rename(columns={'KMEANS_CLASSES':'freq_group'}, inplace=True)

    dist_mat_comp_batdetect2_df = batdetect2_df.copy()
    association_mat = np.ones((len(dist_mat_comp_batdetect2_df), len(dist_mat_comp_batdetect2_df)), dtype='bool')
    for index in range(len(dist_mat_comp_batdetect2_df)):
        row = dist_mat_comp_batdetect2_df.iloc[index]
        dist_to_all_calls = ((dist_mat_comp_batdetect2_df['peak_frequency_time_SPECTROGRAM'] - row['peak_frequency_time_SPECTROGRAM']).values)

        considered_inds = np.where(np.abs(dist_to_all_calls)<=OVERLAP_TIME_THRESHOLD)[0]
        considered_dets = batdetect2_df.iloc[considered_inds]

        det_choices = np.zeros(len(considered_inds))
        likely_call_ind = considered_dets['det_prob'].argmax()

        det_choices[likely_call_ind] = 1
        association_mat[considered_inds, index] = det_choices

    reduced_overlaps_batdetect2_df = batdetect2_df[np.logical_and.reduce(association_mat, axis=1)]
    reduced_overlaps_batdetect2_df.to_csv(f'{BD2_DETS_SAVE_DIR}/{save_loc.stem}_REDUCED_OVERLAPS.csv')
    ravenpro_df = conversion.convert_bd2df_ravenpro(reduced_overlaps_batdetect2_df)
    ravenpro_df.to_csv(f'{BD2_DETS_SAVE_DIR}/{save_loc.stem}_REDUCED_OVERLAPS.txt', sep=RAVENPRO_TABLE_FORMAT, index=False)